[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NU-MSE-LECTURES/465-WINTER2026/blob/main/Week_07/assignments/assignment_07_dpc_ptychography.ipynb)

# Assignment 7: DPC and Ptychography with 4D-STEM

In this assignment, you will analyze a 4D-STEM dataset using two complementary phase-contrast techniques:

1. **Differential Phase Contrast (DPC)** — map beam deflections to reconstruct the projected electric field and electrostatic potential.
2. **Ptychography (ePIE)** — iteratively reconstruct the sample transmission function and probe simultaneously from the full diffraction dataset.

## Learning Objectives
1. Load and inspect a 4D-STEM dataset.
2. Compute center-of-mass (COM) shifts and DPC images from diffraction patterns.
3. Reconstruct the projected electrostatic potential by integrating DPC in Fourier space.
4. Run an ePIE ptychographic reconstruction and interpret the results.
5. Compare DPC and ptychography in terms of resolution, contrast, and sensitivity.

---
## Dataset
- **DPC Part:** SmB$_6$ 4D-STEM data in `data_4dstem/SmB6/` (same as previous assignments).
- **Ptychography Part:** MoS$_2$ 4D-STEM data in `Week_07/lectures/MoS2_4D_data_step2x_crop_3x3uc.mat`.

---

## Part 1: Setup & Data Loading

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
import os

# py4DSTEM for loading 4D-STEM data
import py4DSTEM

print(f"py4DSTEM version: {py4DSTEM.__version__}")
print(f"NumPy version: {np.__version__}")

py4DSTEM version: 0.14.18
NumPy version: 2.2.6


### Load the SmB₆ 4D-STEM Dataset

We use the same SmB₆ dataset from previous assignments. The master HDF5 file links all partial data files.

In [ ]:
# Path to the SmB6 4D-STEM master file
# Adjust the path if needed relative to your environment
data_dir = "../../data_4dstem/SmB6"
master_file = os.path.join(data_dir, "SmB6_10us_1kpx_HM_4_master.h5")

print("Loading 4D-STEM datacube...")
datacube = py4DSTEM.import_file(master_file)
data = datacube.data  # shape: (Rx, Ry, Qx, Qy)

Rx, Ry, Qx, Qy = data.shape
print(f"Datacube shape: ({Rx}, {Ry}, {Qx}, {Qy})")
print(f"  Scan grid:      {Rx} x {Ry} positions")
print(f"  Detector size:  {Qx} x {Qy} pixels")

# Estimated beam center (detector center)
beam_center = (Qx // 2, Qy // 2)
print(f"Approximate beam center: {beam_center}")

Loading 4D-STEM datacube...
Datacube shape: (1000, 1000, 96, 96)
  Scan grid:      1000 x 1000 positions
  Detector size:  96 x 96 pixels
Approximate beam center: (48, 48)


### Inspect the Data

Before any analysis, always visualize a representative diffraction pattern and the bright-field (BF) virtual image.

In [ ]:
# Plot a representative diffraction pattern from the center of the scan
rx_mid, ry_mid = Rx // 2, Ry // 2
dp_example = data[rx_mid, ry_mid, :, :]

# Bright-field virtual image: sum intensity inside a small disk at the beam center
bf_radius = min(Qx, Qy) // 8
Y, X = np.ogrid[:Qy, :Qx]
bf_mask = (X - beam_center[0])**2 + (Y - beam_center[1])**2 <= bf_radius**2
bf_image = np.sum(data * bf_mask[np.newaxis, np.newaxis, :, :], axis=(2, 3))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(np.log1p(dp_example), cmap='inferno')
axes[0].set_title(f'Log-scale diffraction pattern at ({rx_mid}, {ry_mid})')
axes[0].set_xlabel('$q_x$ (pixels)')
axes[0].set_ylabel('$q_y$ (pixels)')

axes[1].imshow(bf_image, cmap='gray')
axes[1].set_title(f'Virtual Bright-Field Image (BF radius = {bf_radius} px)')
axes[1].set_xlabel('Scan X')
axes[1].set_ylabel('Scan Y')

plt.tight_layout()
plt.show()

---
## Part 2: Center-of-Mass (COM) Calculation

**TASK 1:** Compute the COM shift for every scan position.

The COM for a diffraction pattern $I(q_x, q_y)$ is:

$$
q_{x,\text{COM}} = \frac{\sum_{i,j} I(i,j)\cdot i}{\sum_{i,j} I(i,j)}, \quad
q_{y,\text{COM}} = \frac{\sum_{i,j} I(i,j)\cdot j}{\sum_{i,j} I(i,j)}
$$

The **COM shift** is the difference from the undeflected beam center:
$$
\Delta q_x = q_{x,\text{COM}} - c_x, \quad \Delta q_y = q_{y,\text{COM}} - c_y
$$

**Steps:**
1. Create a circular mask to limit the COM calculation to within the bright-field disk.
2. Build coordinate arrays `qx_coords` and `qy_coords`.
3. Loop over all scan positions and compute COM.
4. Subtract the beam center to get the shift arrays `com_x` and `com_y`.

In [ ]:
# --- Create a circular mask around the bright-field disk ---
# Use the same bf_mask defined above but with a slightly larger radius
com_radius = min(Qx, Qy) // 4  # cover the BF disk out to ~1/4 of detector size
Y, X = np.ogrid[:Qy, :Qx]
com_mask = (X - beam_center[0])**2 + (Y - beam_center[1])**2 <= com_radius**2

# --- Build coordinate arrays ---
# qx_coords[i, j] = i  (the x-index of pixel (i,j))
# qy_coords[i, j] = j  (the y-index of pixel (i,j))
qy_coords, qx_coords = np.meshgrid(np.arange(Qy), np.arange(Qx), indexing='ij')

# Apply mask to coordinate arrays (zero out pixels outside the mask)
qx_masked = qx_coords * com_mask
qy_masked = qy_coords * com_mask

# --- Compute COM for all scan positions (vectorized) ---
# Hint: use np.sum over axes=(2,3) to sum over the detector dimensions at once
data_masked = data * com_mask[np.newaxis, np.newaxis, :, :]   # apply mask to all patterns

total_intensity = np.sum(data_masked, axis=(2, 3))             # shape: (Rx, Ry)
total_intensity = np.where(total_intensity == 0, 1, total_intensity)  # avoid division by zero

# TODO: Compute the raw COM position for x and y
# com_x_raw = np.sum(data_masked * qx_masked[np.newaxis, np.newaxis, :, :], axis=(2, 3)) / total_intensity
# com_y_raw = np.sum(data_masked * qy_masked[np.newaxis, np.newaxis, :, :], axis=(2, 3)) / total_intensity
com_x_raw = np.sum(data_masked * qx_masked[np.newaxis, np.newaxis, :, :], axis=(2, 3)) / total_intensity
com_y_raw = np.sum(data_masked * qy_masked[np.newaxis, np.newaxis, :, :], axis=(2, 3)) / total_intensity

# TODO: Subtract beam center to get COM *shift*
# com_x = com_x_raw - beam_center[0]
# com_y = com_y_raw - beam_center[1]
com_x = com_x_raw - beam_center[0]
com_y = com_y_raw - beam_center[1]

print(f"COM calculation complete!")
print(f"  COM_x shift range: {com_x.min():.3f} to {com_x.max():.3f} pixels")
print(f"  COM_y shift range: {com_y.min():.3f} to {com_y.max():.3f} pixels")

### Visualize DPC Signals

**TASK 2:** Plot the DPC images and the vector field of beam deflections.

The DPC signals are simply the COM shifts:
$$
\text{DPC}_x = \Delta q_x, \quad \text{DPC}_y = \Delta q_y, \quad |\text{DPC}| = \sqrt{\Delta q_x^2 + \Delta q_y^2}
$$

Also compute and plot the **divergence** $\nabla \cdot \vec{\text{DPC}}$, which is proportional to the projected charge density.

In [ ]:
# DPC magnitude
dpc_magnitude = np.sqrt(com_x**2 + com_y**2)

# Divergence of the DPC field (finite differences)
# ∇·DPC = ∂(DPC_x)/∂x + ∂(DPC_y)/∂y
# TODO: use np.gradient to compute divergence
div_dpc = np.gradient(com_x, axis=1) + np.gradient(com_y, axis=0)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# DPC_x
im0 = axes[0, 0].imshow(com_x, cmap='RdBu_r')
axes[0, 0].set_title('DPC$_x$ (Horizontal shift)')
plt.colorbar(im0, ax=axes[0, 0], label='Shift (pixels)')

# DPC_y
im1 = axes[0, 1].imshow(com_y, cmap='RdBu_r')
axes[0, 1].set_title('DPC$_y$ (Vertical shift)')
plt.colorbar(im1, ax=axes[0, 1], label='Shift (pixels)')

# DPC magnitude
im2 = axes[1, 0].imshow(dpc_magnitude, cmap='hot')
axes[1, 0].set_title('|DPC| Magnitude')
plt.colorbar(im2, ax=axes[1, 0], label='|Shift| (pixels)')

# Divergence
im3 = axes[1, 1].imshow(div_dpc, cmap='RdBu_r')
axes[1, 1].set_title('Divergence of DPC (∝ projected charge density)')
plt.colorbar(im3, ax=axes[1, 1], label='∇·DPC')

for ax in axes.ravel():
    ax.set_xlabel('Scan X')
    ax.set_ylabel('Scan Y')

plt.suptitle('Differential Phase Contrast — SmB$_6$', fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

---
## Part 3: Phase Reconstruction via Fourier Integration

Since DPC measures the **gradient** of the phase:
$$
\frac{\partial \phi}{\partial x} \propto \text{DPC}_x, \quad \frac{\partial \phi}{\partial y} \propto \text{DPC}_y
$$

we can recover $\phi(\vec{r})$ by solving in Fourier space:
$$
\tilde{\phi}(k_x, k_y) = \frac{i\,k_x\,\widetilde{\text{DPC}_x} + i\,k_y\,\widetilde{\text{DPC}_y}}{k_x^2 + k_y^2}
$$

**TASK 3:** Implement the `integrate_dpc_fourier` function and reconstruct the phase from your DPC maps.

In [ ]:
def integrate_dpc_fourier(dpc_x, dpc_y):
    """
    Integrate DPC phase-gradient maps to recover the phase using Fourier method.

    Parameters
    ----------
    dpc_x, dpc_y : 2D ndarrays
        DPC components (proportional to phase gradients dφ/dx and dφ/dy)

    Returns
    -------
    phase : 2D ndarray
        Reconstructed phase (in arbitrary units)
    """
    Ny, Nx = dpc_x.shape

    # --- Step 1: Build frequency-space coordinate arrays ---
    # np.fft.fftfreq(N) returns frequencies [0, 1/N, ..., (N/2-1)/N, -N/2/N, ..., -1/N]
    kx = np.fft.fftfreq(Nx)   # shape (Nx,)
    ky = np.fft.fftfreq(Ny)   # shape (Ny,)
    KY, KX = np.meshgrid(ky, kx, indexing='ij')  # shape (Ny, Nx)

    # --- Step 2: Compute k² and protect the DC term ---
    k_squared = KX**2 + KY**2
    k_squared[0, 0] = 1  # avoid dividing by zero; DC will be zeroed anyway

    # --- Step 3: Fourier-transform both DPC components ---
    F_dpc_x = np.fft.fft2(dpc_x)
    F_dpc_y = np.fft.fft2(dpc_y)

    # --- Step 4: Integrate in Fourier space ---
    # φ̃(k) = i*(kx*F_DPCx + ky*F_DPCy) / k²
    F_phase = 1j * (KX * F_dpc_x + KY * F_dpc_y) / k_squared
    F_phase[0, 0] = 0  # set DC component to zero (removes arbitrary constant)

    # --- Step 5: Inverse FFT → real-space phase ---
    phase = np.real(np.fft.ifft2(F_phase))

    return phase


# Run the integration
print("Reconstructing phase from DPC via Fourier integration...")
phase = integrate_dpc_fourier(com_x, com_y)

print(f"Phase range: {phase.min():.4f} to {phase.max():.4f} (arb. units)")
print(f"Phase std:   {np.std(phase):.4f}")

In [ ]:
# Plot the reconstructed phase alongside the DPC magnitude for comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(dpc_magnitude, cmap='hot')
axes[0].set_title('|DPC| Magnitude')
plt.colorbar(im0, ax=axes[0], label='|Shift| (pixels)')

im1 = axes[1].imshow(phase, cmap='twilight')
axes[1].set_title('Reconstructed Projected Potential Phase')
plt.colorbar(im1, ax=axes[1], label='Phase (arb. units)')

for ax in axes:
    ax.set_xlabel('Scan X')
    ax.set_ylabel('Scan Y')

plt.suptitle('Phase Reconstruction from DPC — SmB$_6$', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

---
## Part 4: Ptychographic Reconstruction (ePIE) — MoS₂

Now we switch to the MoS₂ dataset and run an **ePIE** (extended Ptychographic Iterative Engine) reconstruction. Unlike DPC, ptychography simultaneously recovers both the **sample transmission function** $O(\vec{r})$ and the **probe** $P(\vec{r})$ by iteratively enforcing:

1. **Real-space constraint:** exit wave = $P(\vec{r}) \cdot O(\vec{r})$
2. **Fourier-space constraint:** $|\mathcal{F}[\psi_{\text{exit}}]|^2 = I_{\text{measured}}(\vec{k})$

The update rules at each iteration $j$ are:
$$
O_{j+1}(\mathbf{r}) = O_j(\mathbf{r}) + \alpha \frac{P_j^*(\mathbf{r})}{|P_j|_{\max}^2}(\psi'_j - \psi_j)
$$
$$
P_{j+1}(\mathbf{r}) = P_j(\mathbf{r}) + \beta \frac{O_j^*(\mathbf{r})}{|O_j|_{\max}^2}(\psi'_j - \psi_j)
$$

where $\alpha$ and $\beta$ are the object and probe update step sizes.

### Required helper modules
The ePIE reconstruction relies on helper code in `Week_07/lectures/helper/ePIE_code/ptycho_recon/`. Make sure it is present before running.

In [2]:
import sys
import time
import scipy.io as sio
from numpy import squeeze, floor, ceil

# Path to the ePIE helper modules (relative to this notebook)
helper_path = "../lectures/helper/"
sys.path.append(helper_path)

import ptycho as pty
import utility_function as utils
import probe as probe_module
import postProcess as postProcess

print("ePIE helper modules loaded successfully.")

ePIE helper modules loaded successfully.


### Load and Preprocess the MoS₂ Dataset

**TASK 4:** Load the `.mat` file and extract the 4D diffraction data and experimental parameters. Then apply the preprocessing steps (background removal and scan-direction transpose).

In [ ]:
# --- Data location ---
data_dir_mos2 = "../lectures/"  # folder containing the .mat file
data_name     = "MoS2_4D_data_step2x_crop_3x3uc.mat"
results_dir   = "./results_ptycho/"
os.makedirs(results_dir, exist_ok=True)

# --- Preprocessing parameters ---
N_roi = 128              # size of square diffraction patterns (pixels)
rot_angle_d = 30         # rotation angle between real and reciprocal space (degrees)
ADU_background_cutoff = 20
ADU_electronCount_ratio = 151.0

# --- Load data ---
data_mos2 = sio.loadmat(data_dir_mos2 + data_name)
print('Variables in .mat file:', list(data_mos2.keys()))

# --- Extract 4D diffraction array and experimental parameters ---
dp = data_mos2['dp'] * 1.0                         # 4D diffraction patterns
voltage       = squeeze(data_mos2['voltage'])       # keV
alpha_max     = squeeze(data_mos2['alpha_max'])     # probe semi-angle (mrad)
df            = squeeze(data_mos2['df'])            # defocus (Å)
cs            = squeeze(data_mos2['cs'])            # spherical aberration (mm)
scanStepSize_x = squeeze(data_mos2['scanStepSize_x'])  # scan step (Å)
scanStepSize_y = squeeze(data_mos2['scanStepSize_y'])
dk            = squeeze(data_mos2['dk'])            # reciprocal pixel size (1/Å)
dx            = 1.0 / dk / N_roi                   # real-space pixel size (Å)

print(f"\nExperimental parameters:")
print(f"  Voltage:      {voltage} keV")
print(f"  Semi-angle:   {alpha_max} mrad")
print(f"  Defocus:      {df} Å")
print(f"  Cs:           {cs} mm")
print(f"  dk:           {dk:.5f} Å⁻¹/pixel")
print(f"  dx:           {dx:.4f} Å/pixel")
print(f"  dp shape:     {dp.shape}")

# --- Preprocessing ---
result_dir_extra = "_preprocessCBED"
dp, result_dir_extra = utils.transpose_cbed(dp, result_dir_extra)
dp, result_dir_extra = utils.background_removal(dp, ADU_background_cutoff, result_dir_extra)

print(f"\nAfter preprocessing, dp shape: {dp.shape}")

### Initialize Probe and Scan Positions

**TASK 5:** Generate the initial STEM probe from the experimental aberration parameters and calculate the scan position coordinates accounting for the rotation angle between real and reciprocal space.

In [ ]:
# --- Build initial probe from aberration parameters ---
probe_init = probe_module.STEMprobe()
probe_init.df        = df
probe_init.cs        = cs
probe_init.alpha_max = alpha_max
probe_init.voltage   = voltage

# --- Calculate scan positions (with rotation correction) ---
N_scan_y = dp.shape[2]
N_scan_x = dp.shape[3]

ppX, ppY, result_dir_extra = utils.calculate_scan_positions(
    N_scan_x, N_scan_y, scanStepSize_x, scanStepSize_y, rot_angle_d, result_dir_extra
)

# Set the object array size large enough to accommodate all probe positions
Ny_max = max([abs(round(np.min(ppY)/dx) - floor(N_roi/2.0)),
              abs(round(np.max(ppY)/dx) + ceil(N_roi/2.0))]) * 2 + 1
Nx_max = max([abs(round(np.min(ppX)/dx) - floor(N_roi/2.0)),
              abs(round(np.max(ppX)/dx) + ceil(N_roi/2.0))]) * 2 + 1
N_image = int(max([Ny_max, Nx_max])) + 20  # add margin for position correction

print(f"Scan positions shape: {ppX.shape}")
print(f"Object array size (N_image): {N_image} x {N_image} pixels")

### Run the ePIE Reconstruction

**TASK 6:** Set the reconstruction parameters and run the ePIE algorithm.

Key parameters:
- `Niter`: number of iterations (start with 20)
- `alpha`: object update step size (try 0.1)
- `beta`: probe update step size (try 1.0)
- `Niter_update_probe`: iteration from which probe update starts (0 = from the beginning)

> **Expected runtime:** ~1–2 minutes for 20 iterations on a modern CPU.

In [ ]:
# --- Set up reconstruction object ---
reconObject = pty.ptycho(dp, dk, probe_init, ppX, ppY)

reconObject.paraDict['dk_y']          = dk
reconObject.paraDict['dk_x']          = dk
reconObject.paraDict['N_image']       = N_image
reconObject.paraDict['N_roi']         = N_roi
reconObject.paraDict['rotationAngle'] = rot_angle_d

# TODO: Experiment with these parameters
reconObject.paraDict['Niter']               = 20   # total iterations
reconObject.paraDict['Niter_update_probe']  = 0    # start probe update from iteration 0
reconObject.paraDict['Niter_save']          = 5    # save every 5 iterations
reconObject.paraDict['alpha']               = 0.1  # object learning rate
reconObject.paraDict['beta']                = 1.0  # probe learning rate
reconObject.paraDict['printID']             = 'MoS2'

# --- Initialize and run ---
result_dir = results_dir + "run1"
result_dir_extra = reconObject.initialize(result_dir)

print("Starting ePIE reconstruction...")
start_time = time.time()
reconObject.recon()
total_time = time.time() - start_time

h, rem = divmod(total_time, 3600)
m, s   = divmod(rem, 60)
print(f"Reconstruction complete! Total time: {int(h):02d}:{int(m):02d}:{int(s):02d}")

### Visualize Reconstruction Convergence

**TASK 7:** Load the saved results from iterations 5, 10, 15, and 20 and plot the reconstructed object phase and probe amplitude at each iteration to assess convergence.

In [ ]:
import scipy.io as sio

Niters = [5, 10, 15, 20]

fig, axes = plt.subplots(2, len(Niters), figsize=(16, 7))

for k, niter in enumerate(Niters):
    # Load the saved checkpoint
    fname = results_dir + f"run1/recon_Niter{niter}.mat"
    recon_data = sio.loadmat(fname)

    obj  = recon_data['object']
    prb  = recon_data['probes']
    px   = recon_data['ppX']
    py   = recon_data['ppY']
    dx_v = recon_data['dx_x']

    # Crop and align the reconstructed object
    obj_crop = postProcess.postProcess(obj, rot_angle_d, px, py, dx_v)

    # Top row: object phase
    axes[0, k].imshow(np.angle(obj_crop), cmap='gray')
    axes[0, k].set_title(f'Object Phase\nIter {niter}', fontsize=10)
    axes[0, k].axis('off')

    # Bottom row: probe amplitude
    axes[1, k].imshow(np.abs(prb[0, :, :]), cmap='hot')
    axes[1, k].set_title(f'Probe Amp.\nIter {niter}', fontsize=10)
    axes[1, k].axis('off')

plt.suptitle('ePIE Convergence — MoS$_2$ Ptychographic Reconstruction', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

---
## Part 5: Self-Reflection

Answer the following questions based on your results:

**1. Center-of-Mass (COM) and DPC:**
- What physical quantity does the DPC signal directly map? What does its divergence represent?
- Why is a circular mask useful when computing the COM? What happens if you include the full detector?

**2. Phase Reconstruction:**
- Why can we not directly measure the phase of the electron wave?
- What artifact might appear at the edges of the Fourier-integrated phase, and how could you mitigate it?

**3. ePIE Ptychography:**
- How does the reconstructed probe compare to an ideal, aberration-free probe? What does this tell you about the microscope?
- Why does ptychography achieve higher resolution than DPC for the same dataset?
- What is the role of the parameters $\alpha$ and $\beta$? What happens if they are too large or too small?

**4. Comparison:**
- In what experimental scenarios would you prefer DPC over ptychography, and vice versa?
- How does the acquisition time/dose requirement differ between the two methods?

### Your Answers:

**1. COM and DPC:**
*Your answer here*

**2. Phase Reconstruction:**
*Your answer here*

**3. ePIE Ptychography:**
*Your answer here*

**4. Comparison:**
*Your answer here*